# Fine-tuning runs on Colab (A100)

Runs `scripts/run_finetune.py` for the configs below on the branch pushed to GitHub, exactly as locally.
Results (`outputs/<name>/`, `reports/results.csv`) and the tile cache are copied to Google Drive after every config,
so a disconnected session loses at most one run.

Before the first run: **Runtime → Change runtime type → A100 GPU**, and add two Colab secrets (🔑 panel on the left):
`KAGGLE_USERNAME`, `KAGGLE_KEY` (from kaggle.json), both with notebook access on.
Hypotheses for the block must be committed **before** these runs (HYPOTHESES.md).

In [ ]:
BRANCH = "part-1"
CONFIGS = [  # one CV run per line; add "--repeats 1 2" to extend a finished run
    "configs/exp/B_resnet18_lr1e-4.yaml",
    "configs/exp/B_resnet18_lr3e-4.yaml",
    "configs/exp/B_resnet50_lr1e-4.yaml",
    "configs/exp/B_resnet50_lr3e-4.yaml",
    "configs/exp/B_effnet_b0_lr1e-4.yaml",
    "configs/exp/B_effnet_b0_lr3e-4.yaml",
    "configs/exp/B_convnext_t_lr1e-4.yaml",
    "configs/exp/B_convnext_t_lr3e-4.yaml",
    "configs/exp/B_convnext_t_dinov3_lr1e-4.yaml",
    "configs/exp/B_convnext_t_dinov3_lr3e-4.yaml",
    "configs/exp/B_resnet18_micronet_lr1e-4.yaml",
    "configs/exp/B_resnet18_micronet_lr3e-4.yaml",
    "configs/exp/B_resnet50_micronet_lr1e-4.yaml",
    "configs/exp/B_resnet50_micronet_lr3e-4.yaml",
    "configs/exp/B_effnet_b0_micronet_lr1e-4.yaml",
    "configs/exp/B_effnet_b0_micronet_lr3e-4.yaml",
]
DRIVE = "/content/drive/MyDrive/soilgsd"
REPO = "/content/repo"

In [ ]:
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
os.makedirs(DRIVE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf {REPO} && git clone -q --branch {BRANCH} https://github.com/lev-laptinov/lab1-cv-acv-soil-grain-size.git {REPO}
%cd {REPO}
!git log -1 --format="%h %s"
# Colab already has torch, numpy, pandas, sklearn, scipy, pillow, matplotlib; soilgsd is imported from src/
!pip install -q timm==1.0.30 tabulate kaggle huggingface_hub
os.environ["PYTHONPATH"] = f"{REPO}/src"

In [ ]:
# Competition data (not committed to git) and the tile cache from an earlier session, if any
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
!kaggle competitions download soil-grain-size-from-photos -p data/raw -q
!cd data/raw && unzip -q -o soil-grain-size-from-photos.zip && rm soil-grain-size-from-photos.zip
!mkdir -p data/cache/tiles && cp -n {DRIVE}/tiles/*.npz data/cache/tiles/ 2>/dev/null; ls -la data/cache/tiles

In [ ]:
# Optional speed / memory check of the first config (no scores)
!python scripts/run_finetune.py {CONFIGS[0]} --smoke 20

In [ ]:
import shutil


def sync_to_drive():
    shutil.copytree("outputs", f"{DRIVE}/outputs", dirs_exist_ok=True)
    shutil.copy("reports/results.csv", f"{DRIVE}/results.csv")
    shutil.copytree("data/cache/tiles", f"{DRIVE}/tiles", dirs_exist_ok=True)


for cfg in CONFIGS:
    print(f"===== {cfg}", flush=True)
    !python scripts/run_finetune.py {cfg}
    sync_to_drive()
print("done; results in", DRIVE)

In [ ]:
import pandas as pd

names = [c.split("/")[-1].split(".yaml")[0] for c in CONFIGS]
res = pd.read_csv("reports/results.csv").set_index("name")
cols = ["backbone", "emd", "emd_std_folds", "emd_fine", "emd_sand", "emd_gravel", "D50_log_err", "repeats", "compute"]
res.loc[res.index.intersection(names), cols].sort_values("emd")